# Doanh thu bị nhân bản: kiểm grain và fan-out trước khi tin dashboard

Head of Merchandising gửi ảnh dashboard: **doanh thu theo sản phẩm năm 2011 cộng lại gần 12 triệu GBP**, trong khi
Finance báo **10.67 triệu**. Dashboard đó join bảng dòng hàng với một "bảng sản phẩm" do ai đó dựng bằng
`SELECT DISTINCT StockCode, Description`. Cùng tuần, một analyst khác báo "giá trị đơn trung bình 20 GBP" trong khi
Sales nói đơn trung bình hơn 500 GBP.

Cả hai lỗi là lỗi **grain**. Bộ `online-retail-ii`, sheet `Year 2010-2011` (541 910 dòng, 01/12/2010 tới
09/12/2011). Việc của bạn: viết bốn hàm kiểm khoá, đo fan-out và tính metric ở đúng grain, chạy chúng trên dữ liệu
thật để định lượng từng lỗi, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/09-analytics-data-modeling/exercises/basic/03-fanout-revenue-audit -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (lần đầu 30-60 giây, sau đó đọc cache)
import os
import sys
import tempfile
import uuid
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure


def load_retail_2010_2011() -> pd.DataFrame:
    """Sheet Year 2010-2011 nguyên trạng + source_row; cache parquet trong thư mục tạm của hệ điều hành."""
    xlsx = ensure("online-retail-ii") / "online_retail_II.xlsx"
    st = xlsx.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"m09_2010-2011_{st.st_mtime_ns}_{st.st_size}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    df.insert(0, "source_row", range(1, len(df) + 1))
    for col in ["Invoice", "StockCode", "Description", "Country"]:
        df[col] = df[col].map(lambda x: None if pd.isna(x) else str(x))  # xlsx trộn int và str
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_suffix(f".{uuid.uuid4().hex}.tmp")
    df.to_parquet(tmp)
    os.replace(tmp, cache)
    return df


raw = load_retail_2010_2011()
print(raw.shape, raw.InvoiceDate.min(), "→", raw.InvoiceDate.max())
raw.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Dòng hợp lệ (có sẵn)

`valid_lines(raw)` áp dụng định nghĩa chung của track: bỏ hoá đơn `C...`, bỏ dòng `Quantity <= 0` hoặc
`Price <= 0`, thêm cột `revenue = Quantity * Price`. Hàm này đã viết sẵn, không phải sửa.

In [ ]:
def valid_lines(raw: pd.DataFrame) -> pd.DataFrame:
    """Dòng bán hợp lệ theo định nghĩa chung + cột revenue. Không sửa raw."""
    invoice = raw["Invoice"].astype(str)
    keep = ~invoice.str.startswith("C") & (raw["Quantity"] > 0) & (raw["Price"] > 0)
    return raw[keep].assign(Invoice=invoice[keep], revenue=lambda d: d["Quantity"] * d["Price"])

## 2. Kiểm một ứng viên khoá

`key_profile(df, keys) -> dict` trả về:

- `rows`: số dòng của `df`; `distinct_keys`: số giá trị khoá khác nhau (NaN tính là một giá trị);
- `duplicated_keys`: số giá trị khoá xuất hiện ở hơn một dòng; `max_rows_per_key`: số dòng nhiều nhất của một khoá;
- `is_unique`: `True` khi mỗi khoá đúng một dòng.

Đây là phiên bản pandas của test `unique` trong dbt. Chạy nó trên **mọi** bảng trước khi join.

In [ ]:
def key_profile(df: pd.DataFrame, keys: list[str]) -> dict:
    """Kiểm một ứng viên khoá: rows, distinct_keys, duplicated_keys, max_rows_per_key (int), is_unique (bool)."""
    # TODO: đếm số dòng mỗi giá trị khoá bằng groupby(keys, dropna=False).size()
    # TODO: duplicated_keys = số giá trị khoá có hơn 1 dòng
    raise NotImplementedError

In [ ]:
# explore: ba ứng viên khoá của bảng dòng hàng, và "bảng sản phẩm" của dashboard
lines = valid_lines(raw)
print(f"{len(lines):,} dòng hợp lệ, doanh thu {lines.revenue.sum():,.2f} GBP")
for keys in (["source_row"], ["Invoice", "StockCode"], ["Invoice", "StockCode", "Quantity", "Price"]):
    print(keys, key_profile(lines, keys))

naive_dim = lines[["StockCode", "Description"]].drop_duplicates()
print("naive_dim theo StockCode:", key_profile(naive_dim, ["StockCode"]))
worst = naive_dim.StockCode.value_counts().idxmax()
naive_dim[naive_dim.StockCode == worst]

## 3. Đo fan-out của một phép join

`join_inflation(left, right, on, measure) -> dict`: left join `right` vào `left` theo `on`, so **số dòng** và
**tổng cột `measure` của left** trước và sau join. Trả `rows_before`, `rows_after`, `before`, `after`,
`inflation = after / before - 1`. Join many-to-one đúng cho `inflation == 0`. Cột trùng tên ở `right` (ngoài
khoá) nhận hậu tố `_right` để cột `measure` của left giữ nguyên tên.

In [ ]:
def join_inflation(left: pd.DataFrame, right: pd.DataFrame, on: str | list[str], measure: str) -> dict:
    """Tổng `measure` (cột của left) trước và sau left join với right.

    Trả rows_before, rows_after (int), before, after, inflation = after / before - 1 (float).
    """
    # TODO: merged = left.merge(right, on=on, how="left", suffixes=("", "_right"))
    # TODO: so số dòng và tổng measure trước/sau
    raise NotImplementedError

In [ ]:
# explore: dashboard sản phẩm join vào naive_dim; fan-out không đều giữa các quốc gia
res = join_inflation(lines, naive_dim, "StockCode", "revenue")
print({k: round(v, 4) for k, v in res.items()})

joined = lines.merge(naive_dim, on="StockCode", suffixes=("", "_dim"))
by_country = (joined.groupby("Country").revenue.sum() / lines.groupby("Country").revenue.sum() - 1)
top = lines.groupby("Country").revenue.sum().nlargest(12).index
fig, ax = plt.subplots(figsize=(9, 4))
by_country[top].sort_values().plot.barh(ax=ax, color="#9aa5b1")
ax.axvline(res["inflation"], color="#f58518", ls="--", lw=1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Doanh thu bị đếm thừa khi join naive_dim, 12 quốc gia lớn nhất", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
print("85123A trước/sau:", lines.loc[lines.StockCode == "85123A", "revenue"].sum(),
      joined.loc[joined.StockCode == "85123A", "revenue"].sum())

## 4. Metric ở grain hoá đơn

- `invoice_header(lines) -> DataFrame`: một dòng mỗi hoá đơn với `Invoice`, `Customer ID`, `Country` (giá trị đầu
  tiên trong hoá đơn), `invoiced_at` (giờ **sớm nhất**: 42 hoá đơn có dòng ghi lệch một phút), `n_lines` (số dòng),
  `invoice_total` (tổng revenue). Sắp theo `Invoice`, index `0..n-1`. Khách vãng lai giữ `Customer ID` rỗng.
- `aov(lines) -> float`: tổng revenue chia số hoá đơn khác nhau.

Sau đó cell explore làm lại lỗi thứ hai: gắn `invoice_total` (grain hoá đơn) vào từng dòng hàng rồi cộng.

In [ ]:
def invoice_header(lines: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi hoá đơn: Invoice, Customer ID, Country, invoiced_at (giờ sớm nhất), n_lines, invoice_total.

    Sắp theo Invoice, index 0..n-1. `lines` là kết quả của valid_lines (có cột revenue).
    """
    # TODO: groupby("Invoice", as_index=False).agg(...)
    raise NotImplementedError


def aov(lines: pd.DataFrame) -> float:
    """Giá trị đơn trung bình: tổng revenue / số hoá đơn khác nhau."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: AOV đúng, "AOV" sai, và tổng hoá đơn bị cộng lặp
header = invoice_header(lines)
print(f"{len(header):,} hoá đơn; n_lines trung bình {header.n_lines.mean():.1f}, trung vị {header.n_lines.median():.0f},"
      f" lớn nhất {header.n_lines.max():,}")
print(f"aov = {aov(lines):.2f} GBP; trung bình revenue mỗi dòng = {lines.revenue.mean():.2f} GBP")
res_h = join_inflation(header, lines[["Invoice", "StockCode"]], "Invoice", "invoice_total")
print(f"gắn tổng hoá đơn vào từng dòng rồi cộng: {res_h['after']:,.0f} GBP, gấp {res_h['after'] / res_h['before']:.2f} lần")

# dòng "trùng" có thật là trùng? hoá đơn có một mã lặp nhiều nhất
dup = lines.groupby(["Invoice", "StockCode"]).size().idxmax()
lines[(lines.Invoice == dup[0]) & (lines.StockCode == dup[1])][["source_row", "Quantity", "Price", "InvoiceDate"]].head()

## 5. Đọc kết quả, rồi làm trắc nghiệm

Trước khi mở app, trả lời cho mình:

1. Dashboard sản phẩm đếm thừa bao nhiêu phần trăm? Trong 12 quốc gia lớn nhất, vì sao mức đếm thừa chênh nhau
   (Japan thấp nhất, Spain cao nhất)? Nếu Merchandising so các quốc gia trên dashboard đó thì sai thế nào?
2. Sửa ở đâu: ở dashboard, hay ở bảng sản phẩm? Test nào lẽ ra phải chặn lỗi này?
3. "20 GBP" và "534 GBP" trả lời hai câu hỏi nào? Câu nào là AOV?
4. 20 dòng cùng hoá đơn 555524, cùng mã 22698, mỗi dòng 1 cái: là lỗi trùng hay cách quầy quét hàng?
   Ai xác nhận được?

Xong thì mở app, làm phần **trắc nghiệm** của bài.